In [30]:
from enum import Enum
import time
import numpy as np
import pandas as pd
import tqdm as notebook_tqdm
import argparse
import csv
import datetime
import json
import os
import shutil
import utils
from constants import *
from run import *
from models import *
from sim import *

In [31]:
"""import AA file for use."""
csv_file_path = "/Users/juar705/Downloads/mock_data.csv"
df = pd.read_csv(csv_file_path)
truncated_df = df.head(501)
del truncated_df['environment']

In [32]:
"""seperate into X_train and y_train sets
    X_train will be the amino acid columns 
    y_train will be the growth column"""

AA_columns = AA_SHORT
growth_columns = 'growth'

X_train = truncated_df[AA_columns].to_numpy()
y_train = truncated_df[growth_columns].to_numpy()

In [33]:
"""Train the models on the data from all previous rounds (excluding Round 1)"""

with open('config.json', 'r') as file:
    config = json.load(file)
    
n_ingredients = len(AA_SHORT)
MODEL_TYPE = ModelType(config["model_type"])   
TRANSFER_MODEL_FOLDER = config.get("transfer_model_folder", None)
transfer_models = transfer_model.models if TRANSFER_MODEL_FOLDER else []

MODEL_TYPE == ModelType.NEURAL_NET
transfer_model = NeuralNetModel.load_trained_models(TRANSFER_MODEL_FOLDER)
N_BAGS = config.get("n_bags", 25)

EXPT_FOLDER = config["experiment_path"]
MODEL_TYPE = ModelType(config["model_type"])
new_round_folder = os.path.join(EXPT_FOLDER, f"Round 1")
models_folder = os.path.join(new_round_folder, f"nn_models")
model = NeuralNetModel(models_folder)

model.train(
            X_train,
            y_train,
            n_ingredients=n_ingredients,
            n_bags=N_BAGS,
            bag_proportion=1.0,
            epochs=50,
            batch_size=20,
            lr=0.001,
            transfer_models=transfer_models,
        )


Bag 0, p=1.00
	EPOCH  1/50 | Train Loss: 0.2344, Train MSE: 0.2344
	EPOCH  2/50 | Train Loss: 0.1373, Train MSE: 0.1373
	EPOCH  3/50 | Train Loss: 0.1103, Train MSE: 0.1103
	EPOCH  4/50 | Train Loss: 0.0803, Train MSE: 0.0803
	EPOCH  5/50 | Train Loss: 0.0679, Train MSE: 0.0679
	EPOCH  6/50 | Train Loss: 0.0610, Train MSE: 0.0610
	EPOCH  7/50 | Train Loss: 0.0534, Train MSE: 0.0534
	EPOCH  8/50 | Train Loss: 0.0451, Train MSE: 0.0451
	EPOCH  9/50 | Train Loss: 0.0390, Train MSE: 0.0390
	EPOCH 10/50 | Train Loss: 0.0389, Train MSE: 0.0389
	EPOCH 11/50 | Train Loss: 0.0309, Train MSE: 0.0309
	EPOCH 12/50 | Train Loss: 0.0275, Train MSE: 0.0275
	EPOCH 13/50 | Train Loss: 0.0218, Train MSE: 0.0218
	EPOCH 14/50 | Train Loss: 0.0174, Train MSE: 0.0174
	EPOCH 15/50 | Train Loss: 0.0148, Train MSE: 0.0148
	EPOCH 16/50 | Train Loss: 0.0151, Train MSE: 0.0151
	EPOCH 17/50 | Train Loss: 0.0118, Train MSE: 0.0118
	EPOCH 18/50 | Train Loss: 0.0125, Train MSE: 0.0125
	EPOCH 19/50 | Train Loss: 0.01

In [34]:
""" Create an array of ones for down direction or 0 for up direction"""

batch_size = config["batch_size"]
DIRECTION = SimDirection(config["direction"])


def media_array(n_ingredients,direction):
    if DIRECTION == SimDirection.DOWN:
        media = np.ones(n_ingredients)
        direction = SimDirection.DOWN
    elif DIRECTION == SimDirection.UP:
        media = np.zeros(n_ingredients)
        direction = SimDirection.UP
    else:
        raise ValueError("Error") 
    return media

media = media_array(n_ingredients, DIRECTION)
print(f"Media Array:\n{media}")

Media Array:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [35]:
def perform_simulations(
    model,
    state,
    n,
    threshold,
    sim_type,
    sim_direction,
    new_round_n,
    unique=False,
    batch_set=None,
    timeout=None,
    n_rollout_trajectories=1,
    go_beyond_frontier=True,
):
    """Performs simulations and generate a batch of experiments to determine the
    'growth frontier' of a bacteria. The simulations determine available actions
    and chooses the next best action to take from the current state. Depending on the
    simulation type, this method differs. If there are no actions that result in a
    predicted growth, the simulation terminates and adds the desired state to the
    batch to test.

    Parameters
    ----------
    model : models.Model
        The model used when running the simulation.
    state : np.ndarray()
        The starting state of the media.
    n : Int
        The number of simulations to perform for this batch.
    threshold : float
        The grow/no grow threshold used to determine when to terminate
        a rollout simulation.
    sim_type : SimType
        The type of simulations to run.
    sim_direction : SimDirection
        The directions in which the simulations run.
    new_round_n : int
        The number of the new round.
    unique : bool, optional
        Take only unique states for the batch, by default False
    batch_set : set(tuple), optional
        The current states already in the batch, by default None
    timeout : int, optional
        The timeout length before forced temination of the simulations
        in seconds, by default None
    n_rollout_trajectories : int, optional
        The number of simulations to perform per state in the rollouts,
        by default 1
    go_beyond_frontier : bool, optional
        Add the state one step beyond the determined 'growth frontier',
        by default True

    Returns
    -------
    pd.DataFrame
        The batch of experiments to perform, where each row is a state
        to test and their associated metadata (simulation parameters, predicted
        growth, etc.)
    """
    state = state.astype(int)
    if batch_set == None:
        batch_set = set()
    batch = []
    batch_frontier_types = []
    terminating_growths = []
    terminating_variances = []

    desc = f"Performing {sim_type.name} Sims ({sim_direction.name})"
    tq = tqdm(total=n, desc=desc)
    not_timed_out = True
    start_time = time.time()
    loops = 1
    n_found_but_exists = 0
    adaptive_choice_history = []

    while len(batch) < n and not_timed_out:
        tq.desc = f"{desc} ({loops} loops)"
        current_state = state.copy()
        current_grow_pred = 0
        current_grow_var = 0
        while (current_state == sim_direction.target_value()).sum() >= 0:
            #print(f"Current state: {current_state}")
            choices = np.argwhere(current_state == sim_direction.target_value())[:, 0]
            #print(choices)
            if choices.size == 0:
                break

            candidate_states = np.tile(current_state, (choices.size, 1))
            if sim_type == SimType.RANDOM:
                action = np.random.choice(choices, 1, False)# Random leave-one-out
                candidate_states[
                    0, action
                ] = sim_direction.action_value()  # Take action
                candidate_states = candidate_states[0].reshape((1, -1))  # Reshape to 2D
                choices = [action]
              

            elif sim_type == SimType.GREEDY:
                # Take all leave-one-out actions
                candidate_states[
                    np.arange(choices.size), choices
                ] = sim_direction.action_value()

            elif sim_type == SimType.ROLLOUT or sim_type == SimType.ROLLOUT_PROB:
                # Take all leave-one-out actions
                rollout_results = np.zeros(choices.size)
                candidate_states[
                    np.arange(choices.size), choices
                ] = sim_direction.action_value()

                # Perform rollouts
                rollout_results = rollout_trajectory(
                    model,
                    candidate_states,
                    n_rollout_trajectories,
                    threshold,
                    sim_direction,
                )
                if sim_type == SimType.ROLLOUT_PROB:
                    # Pick an action idx from a distribution based on softmax of rollout results
                    k = compute_adaptive_choice_const(
                        current_state, sim_direction, n_found_but_exists
                    )
                    adaptive_choice_history.append((k, n_found_but_exists))

                    # Weighted softmax
                    p = utils.softmax(rollout_results, k=k)
                    action_idx = np.random.choice(choices.size, 1, p=p)[0]
                else:
                    # Pick highest predicted reward (mean # removed)
                    action_idx = np.argsort(rollout_results)[-1]

                action = choices[action_idx]
                candidate_states = candidate_states[action_idx].reshape((1, -1))
                choices = [action]
    
            # Get growth prediction of candidate states
            results, results_vars = model.evaluate(candidate_states)
            # Pick highest predicted growth as best action
            best_action_idx = np.argsort(results)[-1]
            best_action = choices[best_action_idx]

            # Keep track of prev state values
            old_state = current_state.copy()
            old_growth_result = current_grow_pred
            old_growth_var = current_grow_var

            # Set new state values
            new_state = current_state.copy()
            new_growth_result = float(results[best_action_idx])
            new_growth_var = float(results_vars[best_action_idx])
            new_state[best_action] = sim_direction.action_value()  # Take best action

            is_down = sim_direction == SimDirection.DOWN
            grows_present = (results >= threshold).sum() > 0

            if (is_down and grows_present) or (not is_down and not grows_present):
                # Keep going if grows are present and DOWN direction, or
                # Keep going if no grows are present and UP direction

                # Update state values
                current_state = new_state
                current_grow_pred = new_growth_result
                current_grow_var = new_growth_var

            elif (is_down and (not grows_present or new_state.sum() == 0)) or (
                not is_down and (grows_present or new_state.sum() == len(new_state))
            ):
                # If going DOWN terminate if:
                #   - no more grows present or removed all ingredients
                #   - Use old state (last known growth predicted), or
                # If going UP terminate if:
                #   - there are grows present or added all ingredients
                #   - Use new state (first known growth predicted)
                if is_down:
                    f_state, b_state = old_state, new_state
                    f_grow_result, b_grow_result = old_growth_result, new_growth_result
                    f_grow_var, b_grow_var = old_growth_var, new_growth_var
                else:
                    f_state, b_state = new_state, old_state
                    f_grow_result, b_grow_result = new_growth_result, old_growth_result
                    f_grow_var, b_grow_var = new_growth_var, old_growth_var

                if go_beyond_frontier:
                    # Add both the "frontier" and "beyond frontier" states
                    states = [f_state, b_state]
                    growth_preds = [f_grow_result, b_grow_result]
                    var_preds = [f_grow_var, b_grow_var]
                    frontier_types = ["FRONTIER", "BEYOND"]
                else:
                    states = [f_state]
                    growth_preds = [f_grow_result]
                    var_preds = [f_grow_var]
                    frontier_types = ["FRONTIER"]

                for st, gr, va, ft in zip(
                    states, growth_preds, var_preds, frontier_types
                ):
                    key = tuple(st)
                    if key not in batch_set or not unique:
                        batch.append(st)
                        terminating_growths.append(gr)
                        terminating_variances.append(va)
                        batch_frontier_types.append(ft)
                        batch_set.add(key)
                        tq.update()
                        print(f"\n\tADDED: {st} - {ft}")
                        if sim_type == SimType.ROLLOUT_PROB:
                            n_found_but_exists -= 1
                            n_found_but_exists = max(n_found_but_exists, 0)
                    else:
                        if sim_type == SimType.ROLLOUT_PROB:
                            n_found_but_exists += 1
                        print(f"\n\tEXISTS: {st} - {ft}")

                    if len(batch) >= n:
                        break
                break

        if timeout is not None:
            not_timed_out = (time.time() - start_time) <= timeout
        loops += 1

    duration = time.time() - start_time

    tq.close()
    if batch:
        batch = pd.DataFrame(np.vstack(batch))
        batch["type"] = sim_type.name
        batch["direction"] = sim_direction.name
        batch["frontier_type"] = batch_frontier_types
        batch["growth_pred"] = terminating_growths
        batch["var"] = terminating_variances
        batch["is_redo"] = False
        batch["round"] = new_round_n
    else:
        batch = pd.DataFrame()

    k_history = [a[0] for a in adaptive_choice_history]
    count_history = [a[1] for a in adaptive_choice_history]

    if len(k_history):
        k_avg = sum(k_history) / len(k_history)
        count_avg = sum(count_history) / len(count_history)
    else:
        k_avg = "n/a"
        count_avg = "n/a"

    metrics = {
        "k_history": k_history,
        "count_history": count_history,
        "k_avg": k_avg,
        "count_avg": count_avg,
        "total_loops_count": loops - 1,
        "time_to_finish_sec": round(duration, 2),
    }
        
    return batch, batch_set, metrics



In [36]:
def make_batch(
    model,
    media,
    new_round_n,
    batch_size,
    sim_types,
    rollout_trajectories,
    threshold,
    timeout=60,
    unique=True,
    direction=SimDirection.DOWN,
    go_beyond_frontier=True,
    used_experiments=None,
    redo_experiments=None,
):
    """ Make a new BacterAI batch; the main function that calls the simulation loops. """
    sim_types=sim_types
    n_types = len(sim_types)
    n_exps = batch_size // n_types
    batch_set = used_experiments
    sub_batches = []
    all_metrics = {}
    for idx, sim_type in enumerate(sim_types):
        if idx == n_types - 1:
            n_exps = batch_size - sum([len(x) for x in sub_batches])
        print(idx, sim_type, batch_size, n_exps, sum([len(x) for x in sub_batches]))
        batch, batch_set, metrics = perform_simulations(
            model,
            media,
            n_exps,
            threshold,
            sim_type,
            direction,
            new_round_n,
            unique=unique,
            timeout=timeout,
            batch_set=batch_set,
            n_rollout_trajectories=rollout_trajectories,
            go_beyond_frontier=go_beyond_frontier,
        )
        sub_batches.append(batch)
        all_metrics[sim_type.name] = metrics

    batch = pd.concat([redo_experiments] + sub_batches, ignore_index=True)
    return batch, batch_set, all_metrics

In [37]:
""" Trained set for batch if using pre-trained data
    If no pre trained data, use 'None' for used_experiments and redo_experiments"""

trained_set = pd.DataFrame(np.hstack((X_train, y_train.reshape(-1,1))))
used_experiments = set(map(tuple,trained_set.to_numpy()))
batch_data= used_experiments

#set parameters needed for make batch function
model = NeuralNetModel.load_trained_models(EXPT_FOLDER)
sim_types=[SimType(3)]
rollout_trajectories=config["n_rollouts"]
threshold=config['grow_threshold']
timeout=60 * 3
unique=True 
direction = SimDirection(0)
go_beyond_frontier=config['beyond_frontier']

# Make batch the main function that calls to make all simulations from perform simulations function
batch, batch_set, all_metrics = make_batch(
    model=model,
    media=media,
    new_round_n=2,
    batch_size=batch_size,
    sim_types=sim_types,
    rollout_trajectories=rollout_trajectories,
    threshold=threshold,
    timeout=timeout,
    unique=unique,
    direction=direction,
    go_beyond_frontier=go_beyond_frontier,
    used_experiments=None,
    redo_experiments=None,)

0 SimType.ROLLOUT_PROB 20 20 0


Performing ROLLOUT_PROB Sims (DOWN):   0%|                 | 0/20 [00:00<?, ?it/s]/Users/juar705/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3859: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/juar705/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:136: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/juar705/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:4267: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/juar705/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/juar705/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/juar705/BacterAI_


	ADDED: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	ADDED: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1] - BEYOND

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	ADDED: [1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1] - BEYOND

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	ADDED: [1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - BEYOND

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	ADDED: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1] - BEYOND

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	ADDED: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1] - BEYOND

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1] - BEYOND

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	ADDED: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1] - BEYOND

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	ADDED: [1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - BEYOND

	EXISTS: [1 1 1

Performing ROLLOUT_PROB Sims (DOWN) (64 loops): 100%|█| 20/20 [00:00<00:00, 85.53i


	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1] - BEYOND

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	EXISTS: [1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1] - BEYOND

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	EXISTS: [1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1] - BEYOND

	EXISTS: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1] - FRONTIER

	ADDED: [1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1] - BEYOND


In [38]:
print(f"Batch:\n,{batch}")

Batch:
,    0  1  2  3  4  5  6  7  8  9  ...  17  18  19          type  direction  \
0   1  1  1  1  1  1  1  1  1  1  ...   1   1   1  ROLLOUT_PROB       DOWN   
1   1  1  1  1  1  1  1  1  1  1  ...   0   1   1  ROLLOUT_PROB       DOWN   
2   1  1  1  1  1  1  0  1  1  1  ...   1   1   1  ROLLOUT_PROB       DOWN   
3   1  1  0  1  1  1  1  1  1  1  ...   1   1   1  ROLLOUT_PROB       DOWN   
4   1  1  1  1  1  1  1  1  1  1  ...   1   1   1  ROLLOUT_PROB       DOWN   
5   1  1  1  1  1  1  1  1  1  1  ...   1   0   1  ROLLOUT_PROB       DOWN   
6   1  1  1  1  1  1  1  1  1  1  ...   1   1   1  ROLLOUT_PROB       DOWN   
7   1  1  1  1  0  1  1  1  1  1  ...   1   1   1  ROLLOUT_PROB       DOWN   
8   1  1  1  0  1  1  1  1  1  1  ...   1   1   1  ROLLOUT_PROB       DOWN   
9   1  1  1  1  1  1  1  1  1  1  ...   1   1   1  ROLLOUT_PROB       DOWN   
10  1  1  1  1  1  1  1  1  1  1  ...   1   1   1  ROLLOUT_PROB       DOWN   
11  1  1  1  1  1  0  1  1  1  1  ...   1   1   1  ROLLO

In [39]:
print(f"Batch Set:\n{batch_set}")

Batch Set:
{(np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1)), (np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1)), (np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1)), (np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int

In [40]:
print(f"Metrics:\n{all_metrics}")

Metrics:
{'ROLLOUT_PROB': {'k_history': [np.float64(100.0), np.float64(100.0), np.float64(100.0), np.float64(100.0), np.float64(100.0), np.float64(100.0), np.float64(99.97037475951163), np.float64(99.97037475951163), np.float64(99.97037475951163), np.float64(99.7632436739075), np.float64(99.20319148370606), np.float64(98.12157028921563), np.float64(98.12157028921563), np.float64(96.36404443012863), np.float64(93.80049995307294), np.float64(90.33640690022142), np.float64(90.33640690022142), np.float64(85.92428334969262), np.float64(80.57353018734796), np.float64(80.57353018734796), np.float64(80.57353018734796), np.float64(80.57353018734796), np.float64(80.57353018734796), np.float64(74.35670792059064), np.float64(67.41043417251569), np.float64(67.41043417251569), np.float64(59.92957878455384), np.float64(59.92957878455384), np.float64(52.15433079812646), np.float64(44.35090653193372), np.float64(36.787944117144235), np.float64(29.711689563161656), np.float64(23.32361769768771), np.floa